In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# --- 1. Load your existing Numpy arrays ---
# Replace filenames if you named them differently
X = np.load("X_features.npy") # (18105, 128, 94, 1)
y = np.load("y_labels.npy")    # (18105,)

# --- 2. Train/Test Split ---
# 80% Train, 20% Test. 'stratify' ensures balanced classes in both.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

class AudioDataset(Dataset):
    def __init__(self, features, labels):
        # PyTorch expects (Batch, Channel, Height, Width)
        # So we permute (18105, 128, 94, 1) -> (18105, 1, 128, 94)
        self.X = torch.from_numpy(features).permute(0, 3, 1, 2).float()
        self.y = torch.from_numpy(labels).float().unsqueeze(1)
        
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# --- 3. The CNN Architecture ---
class DeepfakeDetector(nn.Module):
    def __init__(self):
        super(DeepfakeDetector, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Adaptive pooling ensures fixed output size for the Linear layer
        self.adaptive_pool = nn.AdaptiveAvgPool2d((8, 8))
        self.fc = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# --- 4. Training Initialization ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepfakeDetector().to(device)
train_loader = DataLoader(AudioDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(AudioDataset(X_test, y_test), batch_size=32)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.BCEWithLogitsLoss()

# --- 5. Training Loop ---
print(f"Training on {device}...")
for epoch in range(15):
    model.train()
    total_loss = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Simple Val Check
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            outputs = model(bx.to(device))
            preds.extend((outputs > 0).float().cpu().numpy())
            actuals.extend(by.numpy())
    
    acc = accuracy_score(actuals, preds)
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {acc:.4f}")

#torch.save(model.state_dict(), "final_audio_model.pth")

Training on cpu...
Epoch 1 | Loss: 0.4558 | Val Acc: 0.8329
Epoch 2 | Loss: 0.3372 | Val Acc: 0.8826
Epoch 3 | Loss: 0.2706 | Val Acc: 0.8520
Epoch 4 | Loss: 0.2347 | Val Acc: 0.6998
Epoch 5 | Loss: 0.2142 | Val Acc: 0.9224
Epoch 6 | Loss: 0.1831 | Val Acc: 0.9379
Epoch 7 | Loss: 0.1640 | Val Acc: 0.6553
Epoch 8 | Loss: 0.1461 | Val Acc: 0.7498
Epoch 9 | Loss: 0.1442 | Val Acc: 0.9304
Epoch 10 | Loss: 0.1395 | Val Acc: 0.6785
Epoch 11 | Loss: 0.1237 | Val Acc: 0.9395
Epoch 12 | Loss: 0.1293 | Val Acc: 0.6244
Epoch 13 | Loss: 0.1215 | Val Acc: 0.6109
Epoch 14 | Loss: 0.1140 | Val Acc: 0.9500
Epoch 15 | Loss: 0.0973 | Val Acc: 0.9638


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# ===============================
# 1. Load Data
# ===============================
X = np.load("X_features.npy")   # (18105, 128, 94, 1)
y = np.load("y_labels.npy")     # (18105,)

# -------------------------------
# Train / Val / Test Split
# -------------------------------
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42
)  # ≈70/15/15

# ===============================
# 2. Dataset Class
# ===============================
class AudioDataset(Dataset):
    def __init__(self, features, labels):
        self.X = torch.from_numpy(features).permute(0, 3, 1, 2).float()
        self.y = torch.from_numpy(labels).float().unsqueeze(1)

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ===============================
# 3. CNN Model
# ===============================
class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((8, 8))

        self.fc = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


# ===============================
# 4. Setup
# ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepfakeDetector().to(device)

train_loader = DataLoader(AudioDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(AudioDataset(X_val, y_val), batch_size=32)
test_loader  = DataLoader(AudioDataset(X_test, y_test), batch_size=32)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.BCEWithLogitsLoss()

# ===============================
# 5. Evaluation Function
# ===============================
def evaluate(model, loader):
    model.eval()
    preds, probs, actuals = [], [], []

    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(device)
            outputs = model(bx)

            prob = torch.sigmoid(outputs).cpu().numpy()
            pred = (prob > 0.5).astype(int)

            probs.extend(prob)
            preds.extend(pred)
            actuals.extend(by.numpy())

    acc = accuracy_score(actuals, preds)
    auc = roc_auc_score(actuals, probs)

    return acc, auc, preds, actuals


# ===============================
# 6. Training Loop (Early Stop)
# ===============================
epochs = 10
patience = 5
best_val_acc = 0
counter = 0

print(f"Training on {device}...\n")

for epoch in range(epochs):

    # ---- Train ----
    model.train()
    total_loss = 0

    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)

        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    # ---- Validate ----
    val_acc, val_auc, _, _ = evaluate(model, val_loader)

    print(f"Epoch {epoch+1:02d} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")

    # ---- Early Stopping ----
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("\nEarly stopping triggered.")
            break


# ===============================
# 7. Final Test Evaluation
# ===============================
print("\nFinal Test Evaluation:")

test_acc, test_auc, test_preds, test_actuals = evaluate(model, test_loader)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC: {test_auc:.4f}\n")

print("Classification Report:")
print(classification_report(test_actuals, test_preds))

Training on cpu...

Epoch 01 | Loss: 0.4627 | Val Acc: 0.8291 | Val AUC: 0.9197
Epoch 02 | Loss: 0.3541 | Val Acc: 0.7829 | Val AUC: 0.9367
Epoch 03 | Loss: 0.2854 | Val Acc: 0.8649 | Val AUC: 0.9664
Epoch 04 | Loss: 0.2438 | Val Acc: 0.6785 | Val AUC: 0.9510
Epoch 05 | Loss: 0.2139 | Val Acc: 0.9472 | Val AUC: 0.9874
Epoch 06 | Loss: 0.1872 | Val Acc: 0.7656 | Val AUC: 0.9665
Epoch 07 | Loss: 0.1716 | Val Acc: 0.7294 | Val AUC: 0.9764
Epoch 08 | Loss: 0.1515 | Val Acc: 0.7796 | Val AUC: 0.9724
Epoch 09 | Loss: 0.1483 | Val Acc: 0.9568 | Val AUC: 0.9919
Epoch 10 | Loss: 0.1306 | Val Acc: 0.7442 | Val AUC: 0.9806

Final Test Evaluation:
Test Accuracy: 0.7618
Test AUC: 0.9794

Classification Report:
              precision    recall  f1-score   support

         0.0       0.62      1.00      0.77      1076
         1.0       1.00      0.61      0.75      1640

    accuracy                           0.76      2716
   macro avg       0.81      0.80      0.76      2716
weighted avg       0.

In [3]:
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_curve, accuracy_score, f1_score, classification_report

def get_probs(model, loader):
    model.eval()
    probs, labels = [], []

    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(device)
            outputs = model(bx)
            p = torch.sigmoid(outputs).cpu().numpy().ravel()

            probs.extend(p)
            labels.extend(by.numpy().ravel())

    return np.array(probs), np.array(labels)


test_probs, test_labels = get_probs(model, test_loader)
val_probs, val_labels = get_probs(model, val_loader)

In [4]:
precision, recall, thresholds = precision_recall_curve(val_labels, val_probs)

f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
best_thresh_f1 = thresholds[best_idx]

print("Best threshold (F1):", best_thresh_f1)
print("Best Val F1:", f1_scores[best_idx])

Best threshold (F1): 0.0048094536
Best Val F1: 0.9461910469957554


In [5]:
fpr, tpr, thresholds_roc = roc_curve(val_labels, val_probs)
j_scores = tpr - fpr

best_idx_j = np.argmax(j_scores)
best_thresh_j = thresholds_roc[best_idx_j]

print("Best threshold (Youden J):", best_thresh_j)

Best threshold (Youden J): 0.009377477690577507


In [7]:
#THIS IS THE MOST IMPROVED MODEL THAT USES ALL OPTIMAL PARAMETERS. USING LOG_MEL AND CNN

In [6]:
final_thresh = best_thresh_f1

test_preds_opt = (test_probs > final_thresh).astype(int)

print("\nImproved Test Metrics (Optimized Threshold):\n")

print("Accuracy:", accuracy_score(test_labels, test_preds_opt))
print("F1 Score:", f1_score(test_labels, test_preds_opt))

print("\nClassification Report:")
print(classification_report(test_labels, test_preds_opt))


Improved Test Metrics (Optimized Threshold):

Accuracy: 0.9300441826215022
F1 Score: 0.9427365883062085

Classification Report:
              precision    recall  f1-score   support

         0.0       0.93      0.89      0.91      1076
         1.0       0.93      0.95      0.94      1640

    accuracy                           0.93      2716
   macro avg       0.93      0.92      0.93      2716
weighted avg       0.93      0.93      0.93      2716

